# Fit the weighted fixed-effects model on index change

Fit the weighted fixed-effects difference-in-differences model to site-level
mean changes from `outputs/change_ci/`, per `configs/change_did.yaml`, for two
model specifications: the **site + period** model (intercept, deviation-coded
site and year-pair effects, both event indicators; 14 parameters, 28 residual
df) and the **period-only** model (the same without site effects; 8
parameters, 34 residual df). Each specification runs the main model at the
primary τ, both placebo tests, and the sensitivity reruns, for every
configured index and month; every output row is labeled with its model.
Results are saved under `outputs/change_did/<INDEX>/`. For the method, see the
[site + period methods](../docs/white-papers/Before-After-Control-Impact-Analysis.md)
and [period-only methods](../docs/white-papers/Period-Only-Control-Impact-Analysis.md).


In [1]:
from pathlib import Path

import pandas as pd

from change_did import (
    ChangeDidAnalysis,
    DidAnalysisConfig,
    flagged_se_report,
    load_change_results,
)

ROOT = Path.cwd().parent
assert (ROOT / "pyproject.toml").exists(), "Run notebooks from the notebooks/ folder"
pd.set_option("display.width", 160)

## Analysis configuration

In [2]:
config = DidAnalysisConfig.from_yaml(
    ROOT / "configs" / "change_did.yaml", project_root=ROOT
)
config.to_dict()

{'input_paths': ['/home/tshihadah/code/wadi-gaza-research/outputs/change_ci/NDVI/results.csv',
  '/home/tshihadah/code/wadi-gaza-research/outputs/change_ci/GNDVI/results.csv',
  '/home/tshihadah/code/wadi-gaza-research/outputs/change_ci/SAVI/results.csv',
  '/home/tshihadah/code/wadi-gaza-research/outputs/change_ci/NDRE/results.csv'],
 'output_dir': '/home/tshihadah/code/wadi-gaza-research/outputs/change_did',
 'indices': ('NDVI', 'GNDVI', 'SAVI', 'NDRE'),
 'primary_index': 'NDVI',
 'months': (1, 5, 9, 11),
 'supplementary_months': (1,),
 'affected_site': 'WadiGaza',
 'event_pairs': [[2023, 2024], [2024, 2025]],
 'models': ('site_period', 'period_only'),
 'tau': 0.05,
 'tau_sensitivity': (0.02, 0.05, 0.1),
 'alpha': 0.05,
 'fill_n_eff': 5,
 'expected_flagged_primary': 22}

## Load and validate the change-CI results

Loading runs the brief's checks: every index × month × τ subset holds one row
per site × year pair with no duplicates, `year_end = year_start + 1`, `se` is
missing exactly where `flag_no_range` is true, and `mean_change` is never
missing. A failed check raises with a clear message.

In [3]:
frame = load_change_results(config)
expected = (
    len(config.indices) * len(config.months) * len(config.tau_sensitivity)
    * frame["study_area"].nunique()
    * frame[["year_start", "year_end"]].drop_duplicates().shape[0]
)
if len(frame) != expected:
    raise ValueError(f"Expected {expected} input rows, loaded {len(frame)}")
print(f"Loaded {len(frame)} rows for {frame['study_area'].nunique()} sites")
frame.head()

Loaded 2016 rows for 7 sites


,study_area,index,month,year_start,year_end,tau,is_primary_tau,engine,n,mean_change,...,se,ci_low,ci_high,naive_se,naive_ci_low,naive_ci_high,ci_excludes_zero,flag_no_range,flag_low_n_eff,error
0,AlAhrashProtectorate,NDVI,1,2019,2020,0.02,False,fft,33848,0.025359,...,0.005149,0.015121,0.035598,0.000259,0.024852,0.025867,True,False,False,NaN
1,AlAhrashProtectorate,NDVI,1,2019,2020,0.05,True,fft,33848,0.025359,...,0.004832,0.015769,0.034950,0.000259,0.024852,0.025867,True,False,False,NaN
2,AlAhrashProtectorate,NDVI,1,2019,2020,0.10,False,fft,33848,0.025359,...,0.004379,0.016687,0.034032,0.000259,0.024852,0.025867,True,False,False,NaN
3,AlAhrashProtectorate,NDVI,1,2020,2021,0.02,False,fft,33848,-0.019612,...,NaN,NaN,NaN,0.000321,-0.020241,-0.018983,False,True,False,NaN
4,AlAhrashProtectorate,NDVI,1,2020,2021,0.05,True,fft,33848,-0.019612,...,0.008229,-0.036136,-0.003088,0.000321,-0.020241,-0.018983,True,False,False,NaN


## Flagged (missing-SE) rows at the primary τ

The brief expects 22 flagged NDVI rows at τ = 0.05 (5 each in May, September
and November; 7 in January); a different NDVI count logs a warning, not an
error. Counts for the other indices are reported for the record.

In [4]:
analysis = ChangeDidAnalysis(config)
flagged = flagged_se_report(frame, config)
flagged

,index,month_01,month_05,month_09,month_11,total
0,NDVI,7,5,5,5,22
1,GNDVI,7,6,3,5,21
2,SAVI,7,5,4,4,20
3,NDRE,9,5,7,4,25


## Run and save

Each model × index × month runs the primary specification plus the sensitivity
variants (τ ∈ {0.02, 0.05, 0.10}, exclude-flagged, n_eff = 5 fill), and the
primary specification adds both placebo tests, fit with the same model as the
run they belong to. Statsmodels OLS residuals and leverages give the Hedges
method-of-moments estimate σ_u² = max(0, [RSS − Σ(1 − hᵢ)SEᵢ²] / [n − k]).
Indicator-saturated cells have leverage one and contribute no residual
measurement variance. Statsmodels WLS uses inverse estimated total variances,
fixed covariance scale = 1, and explicit Student-t inference.


In [5]:
results = analysis.run(frame)
analysis.save(results)
print(
    f"Saved {len(results.main)} main rows, {len(results.effects)} effect rows, "
    f"{len(results.placebo_space)} placebo-space rows, "
    f"and {len(results.placebo_spacetime)} placebo-space-and-time rows under {config.output_dir}"
)
results.main.groupby(["model", "index", "variant"]).size().unstack()

Dropped indicator d_persist for site_period NDVI month 9 variant exclude_flagged: its cell is absent after SE handling


Dropped indicator d_persist for site_period NDVI month 11 variant exclude_flagged: its cell is absent after SE handling


Dropped indicator d_persist for site_period SAVI month 9 variant exclude_flagged: its cell is absent after SE handling


Dropped indicator d_persist for site_period NDRE month 9 variant exclude_flagged: its cell is absent after SE handling


Dropped indicator d_persist for site_period NDRE month 11 variant exclude_flagged: its cell is absent after SE handling


Dropped indicator d_persist for period_only NDVI month 9 variant exclude_flagged: its cell is absent after SE handling


... (2 outputs left out)

Dropped indicator d_persist for period_only NDRE month 9 variant exclude_flagged: its cell is absent after SE handling


Dropped indicator d_persist for period_only NDRE month 11 variant exclude_flagged: its cell is absent after SE handling


Saved 310 main rows, 352 effect rows, 224 placebo-space rows, and 1120 placebo-space-and-time rows under /home/tshihadah/code/wadi-gaza-research/outputs/change_did


variant            exclude_flagged  n_eff_fill  primary  tau_0.02  tau_0.1
model       index                                                         
period_only GNDVI                8           8        8         8        8
            NDRE                 6           8        8         8        8
            NDVI                 6           8        8         8        8
            SAVI                 7           8        8         8        8
site_period GNDVI                8           8        8         8        8
            NDRE                 6           8        8         8        8
            NDVI                 6           8        8         8        8
            SAVI                 7           8        8         8        8

## Primary-specification estimates

One row per model × index × month × term at the primary τ with the largest-SE
fill. `d_event` is the estimate of interest (β, first event pair) and
`d_persist` is the persistence estimate (β′, second event pair). A clipped
σ_u² means the measurement SEs exceed the total scatter. Deviation-coded site
and year-pair effects with their joint Wald F-tests are saved to
`did_effects.csv` and the per-model summary tables.


In [6]:
primary = results.main.loc[results.main["variant"] == "primary"]
display_columns = [
    "model", "index", "role", "month", "supplementary", "term", "estimate",
    "ci_low", "ci_high", "p_value", "sigma_u", "sigma_u_clipped", "df", "n_filled",
]
primary[display_columns].sort_values(
    ["model", "index", "month", "term"]
).reset_index(drop=True)

,model,index,role,month,supplementary,term,estimate,ci_low,ci_high,p_value,sigma_u,sigma_u_clipped,df,n_filled
0,period_only,GNDVI,robustness,1,True,d_event,-0.074606,-0.169747,0.020536,0.120283,0.038523,False,34,7
1,period_only,GNDVI,robustness,1,True,d_persist,-0.032529,-0.125017,0.059960,0.479640,0.038523,False,34,7
2,period_only,GNDVI,robustness,5,False,d_event,-0.051401,-0.093129,-0.009674,0.017267,0.016936,False,34,6
3,period_only,GNDVI,robustness,5,False,d_persist,-0.005025,-0.047990,0.037940,0.813559,0.016936,False,34,6
4,period_only,GNDVI,robustness,9,False,d_event,-0.066952,-0.106621,-0.027284,0.001600,0.012170,False,34,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,site_period,SAVI,robustness,5,False,d_persist,-0.025876,-0.060012,0.008259,0.131703,0.011453,False,28,5
60,site_period,SAVI,robustness,9,False,d_event,-0.079256,-0.111271,-0.047241,0.000023,0.007432,False,28,4
61,site_period,SAVI,robustness,9,False,d_persist,-0.028002,-0.060032,0.004027,0.084133,0.007432,False,28,4
62,site_period,SAVI,robustness,11,False,d_event,-0.071682,-0.106081,-0.037283,0.000204,0.011330,False,28,4


## Placebo p-values per model, index and month


In [7]:
group_keys = ["model", "index", "month"]
space_p = results.placebo_space.groupby(group_keys)["p_value"].first()
spacetime_p = results.placebo_spacetime.groupby(group_keys)["p_value"].first()
ratio = results.placebo_space.groupby(group_keys)["ratio_to_largest_placebo"].first()
pd.DataFrame({
    "placebo_space_p": space_p,
    "ratio_to_largest_placebo": ratio,
    "placebo_spacetime_p": spacetime_p,
}).reset_index()

,model,index,month,placebo_space_p,ratio_to_largest_placebo,placebo_spacetime_p
0,period_only,GNDVI,1,0.142857,0.781198,0.028571
1,period_only,GNDVI,5,0.142857,2.365423,0.028571
2,period_only,GNDVI,9,0.142857,5.513719,0.028571
3,period_only,GNDVI,11,0.142857,3.809255,0.028571
4,period_only,NDRE,1,0.142857,0.704132,0.057143
5,period_only,NDRE,5,0.142857,3.589293,0.028571
6,period_only,NDRE,9,0.142857,7.276885,0.028571
7,period_only,NDRE,11,0.142857,13.428096,0.028571
8,period_only,NDVI,1,0.142857,0.667002,0.057143
9,period_only,NDVI,5,0.142857,3.449247,0.028571


## Validate the DiD pipeline

`tests/test_did.py` and `tests/test_did_config.py` exercise the design
build, the weighted FE fits, and the placebo/sensitivity runs against
synthetic fixtures. Run them here as a final gate; a failure means the
design, the fill, or the σ_u² step has regressed.


In [8]:
import subprocess
import sys

check = subprocess.run(
    [sys.executable, "-m", "pytest",
     str(ROOT / "tests" / "test_did.py"),
     str(ROOT / "tests" / "test_did_config.py"), "-q"],
    capture_output=True, text=True, cwd=ROOT,
)
print(check.stdout[-2000:])
if check.returncode != 0:
    print(check.stderr[-2000:])
    raise RuntimeError("DiD test suite failed; inspect the output above")


............................................................             [100%]
60 passed in 0.48s

